# CMSC 173 &middot; Module 13 &middot; Example 4
## An autoencoder for the digits

A network trained to rebuild its own input through a 2-number bottleneck. The 2 numbers place every digit on a map; points on the map decode into new digits; the digits it rebuilds worst are the unusual ones (module 11).

**How to use this notebook.** Runtime &rarr; Run all. Every number printed below is on the
slides (NumPy and scikit-learn 1.4.2; a newer version can differ slightly for the trained networks);
change something and see what moves.

**Module 10's answer first**: PCA keeps 2 numbers per digit. How much is lost?

In [ ]:
import numpy as np, warnings
from sklearn.datasets import load_digits
from sklearn.decomposition import PCA
from sklearn.neural_network import MLPRegressor
warnings.filterwarnings("ignore")

X, y = load_digits(return_X_y=True); X = X / 16               # 1,797 digits, pixels 0 to 1
pca = PCA(2).fit(X)
print(round(((X - pca.inverse_transform(pca.transform(X))) ** 2).mean(), 4))   # 0.0524

**The autoencoder**: 64 &rarr; 32 &rarr; 2 &rarr; 32 &rarr; 64, tanh neurons. The target is the input itself, so no labels are used.

In [ ]:
ae = MLPRegressor(hidden_layer_sizes=(32, 2, 32), activation="tanh", solver="lbfgs", max_iter=2000,
                  random_state=173).fit(X, X)
err = ((X - ae.predict(X)) ** 2).mean(axis=1)                 # rebuild error of each digit
print(sum(w.size for w in ae.coefs_) + sum(b.size for b in ae.intercepts_), round(err.mean(), 4))   # 4354 0.0292

**The encoder and the decoder**, written out from the network's weights: each layer is module 12's weighted sum and activation.

In [ ]:
W, b = ae.coefs_, ae.intercepts_
encode = lambda X: np.tanh(np.tanh(X @ W[0] + b[0]) @ W[1] + b[1])       # 64 -> 32 -> 2
decode = lambda C: np.tanh(C @ W[2] + b[2]) @ W[3] + b[3]                # 2 -> 32 -> 64
codes = encode(X)
print(codes[:3].round(3), y[:3])

**The map.** Each digit at its 2 numbers, coloured afterwards by the true label.

In [ ]:
import matplotlib.pyplot as plt
plt.figure(figsize=(6, 5))
plt.scatter(codes[:, 0], codes[:, 1], c=y, cmap="tab10", s=6)
for d in range(10):
    plt.text(*np.median(codes[y == d], axis=0), str(d), fontsize=16, weight="bold")
plt.xlabel("code number 1"); plt.ylabel("code number 2"); plt.show()

**Generating.** Decode a 7 &times; 7 grid of points on the map: digits nobody wrote.

In [ ]:
grid = np.linspace(-0.9, 0.9, 7)
fig, ax = plt.subplots(7, 7, figsize=(5, 5))
for r, b2 in enumerate(grid[::-1]):
    for c, b1 in enumerate(grid):
        ax[r, c].imshow(np.clip(decode(np.array([[b1, b2]]))[0], 0, 1).reshape(8, 8), cmap="Greys")
        ax[r, c].axis("off")
plt.show()

**The worst-rebuilt digits** are the unusual ones: module 11's anomaly score, from a network.

In [ ]:
worst = np.argsort(err)[::-1][:6]
print(y[worst], err[worst].round(3), round(np.median(err), 4))
# [2 2 9 4 8 5] [0.092 0.087 0.085 0.085 0.084 0.081] 0.0271
# (scikit-learn 1.4.2; on 1.9 the six differ, errors 0.082 to 0.103, but they still stand out the same way)

**Try it.** Give the code 3 numbers, `hidden_layer_sizes=(32, 3, 32)`. How far does the rebuild error fall?